# Earmuff 3D Vibration Analysis
**ABS Plastic | 1–4 kHz | PINN + ParaView**

> Set runtime to T4 GPU before running

In [ ]:
# Cell 1 — Install & clone repo
!pip install trimesh vtk scipy tqdm -q
!git clone https://github.com/YOUR_USERNAME/earmuff-pinn.git
%cd earmuff-pinn
import sys; sys.path.insert(0, '.')
print('✅ Ready')

In [ ]:
# Cell 2 — Upload STL
import os
os.makedirs('/content/outputs', exist_ok=True)

from google.colab import files as colab_files
uploaded = colab_files.upload()

STL_PATH = None
for fname, data in uploaded.items():
    STL_PATH = f'/content/{fname}'
    open(STL_PATH, 'wb').write(data)
    print(f'Uploaded: {STL_PATH}')

In [ ]:
# Cell 3 — Load mesh + build occupancy grid
import gc
import numpy as np
import matplotlib.pyplot as plt
from geometry.mesh_utils import load_mesh, build_occupancy_grid

mesh = load_mesh(STL_PATH)
BBOX_MIN     = mesh.bounds[0]
BBOX_MAX     = mesh.bounds[1]
BBOX_EXTENTS = BBOX_MAX - BBOX_MIN

print(f'Vertices  : {len(mesh.vertices):,}')
print(f'Watertight: {mesh.is_watertight}')
print(f'Bounds    : {(BBOX_MIN*100).round(2)} → {(BBOX_MAX*100).round(2)} cm')
print(f'Volume    : {mesh.volume*1e6:.2f} cm³')

# mesh preview
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
v = mesh.vertices[::max(1, len(mesh.vertices)//3000)]
for ax, (xi, yi, xl, yl) in zip(axes, [(0,2,'x','z'),(1,2,'y','z'),(0,1,'x','y')]):
    ax.scatter(v[:,xi]*100, v[:,yi]*100, s=0.5, c='steelblue', alpha=0.5)
    ax.set_xlabel(f'{xl} [cm]'); ax.set_ylabel(f'{yl} [cm]')
    ax.set_aspect('equal')
plt.tight_layout()
plt.savefig('/content/outputs/stl_preview.png', dpi=120)
plt.show()

OCC_GRID, occ_xs, occ_ys, occ_zs, _ = build_occupancy_grid(mesh, res=64)
print(f'Occupancy grid: {OCC_GRID.shape} | Inside: {OCC_GRID.sum():,}')

In [ ]:
# Cell 4 — Config
import torch
from core.config import (
    C_ABS, RHO_ABS, F_NOM, K, LEAKAGE_DB, LEAKAGE_FACTOR,
    N_INTERIOR, N_BOUNDARY, N_SEAL, N_LEAKAGE, EPOCHS, LR, DEVICE
)

PAD     = 0.05
DOM_MIN = BBOX_MIN - PAD * BBOX_EXTENTS
DOM_MAX = BBOX_MAX + PAD * BBOX_EXTENTS

print(f'Device     : {DEVICE}')
print(f'k          : {K:.4f} rad/m  |  λ = {C_ABS/F_NOM*100:.2f} cm')
print(f'Leakage    : {LEAKAGE_FACTOR:.4f}  ({LEAKAGE_DB} dB)')

In [ ]:
# Cell 5 — Sample points
from geometry.sampler import sample_interior, sample_surface, sample_seal, sample_leakage

pts_interior = sample_interior(mesh, N_INTERIOR, DOM_MIN, DOM_MAX)
pts_surface  = sample_surface(mesh, N_BOUNDARY)
pts_seal     = sample_seal(mesh, N_SEAL, BBOX_MIN, BBOX_EXTENTS)
pts_leakage  = sample_leakage(mesh, N_LEAKAGE, BBOX_MIN, BBOX_EXTENTS)
gc.collect()

check = mesh.contains(pts_interior[:200])
print(f'Interior quality: {check.sum()}/200 inside mesh')
print(f'Interior: {len(pts_interior):,} | Surface: {len(pts_surface):,} | Seal: {len(pts_seal):,} | Leakage: {len(pts_leakage):,}')

In [ ]:
# Cell 6 — Build model
import numpy as np
from core.model import EarmuffPINN

model = EarmuffPINN(layers=6, width=128).to(DEVICE)
all_pts = np.vstack([pts_interior, pts_surface])
model.set_normalization(all_pts)
del all_pts; gc.collect()

print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# Cell 7 — Train
from training.trainer import train

history = train(model, pts_interior, pts_surface, pts_seal, pts_leakage)
print(f'Final loss: {history["total"][-1]:.6f}')

In [ ]:
# Cell 8 — Loss curves
from visualization.plots import plot_loss
plot_loss(history, EPOCHS, '/content/outputs/training_loss.png')

In [ ]:
# Cell 9 — Inference
from training.inference import run_inference

U_vol, S_vol, inside_3d, xs, ys, zs = run_inference(
    model, DOM_MIN, DOM_MAX,
    OCC_GRID, occ_xs, occ_ys, occ_zs, BBOX_MIN,
    grid_res=40
)
print(f'u_max={U_vol[inside_3d].max():.4f} | stress_max={S_vol[inside_3d].max():.4f}')

In [ ]:
# Cell 10 — Contour maps
import os
from visualization.plots import plot_contours
plot_contours(U_vol, S_vol, xs, ys, zs, mesh,
              os.path.basename(STL_PATH),
              '/content/outputs/vibration_contours.png')

In [ ]:
# Cell 11 — Frequency sweep
import os
from visualization.plots import plot_frequency_sweep
plot_frequency_sweep(U_vol, xs, ys, zs, mesh,
                     os.path.basename(STL_PATH),
                     '/content/outputs/frequency_sweep.png')

In [ ]:
# Cell 12 — Export for ParaView
from visualization.vtk_export import export_vti, export_vtk_legacy
export_vti(U_vol, S_vol, xs, ys, zs, '/content/outputs/earmuff_displacement.vti')
export_vtk_legacy(U_vol, S_vol, xs, ys, zs, '/content/outputs/earmuff_displacement.vtk')

In [ ]:
# Cell 13 — Download
from google.colab import files as colab_files
for f in ['earmuff_displacement.vti', 'earmuff_displacement.vtk',
          'vibration_contours.png', 'frequency_sweep.png', 'training_loss.png']:
    p = f'/content/outputs/{f}'
    if os.path.exists(p):
        colab_files.download(p)